# Prepare Beauty Atomic Files


In [1]:
from pathlib import Path
import pandas as pd


In [2]:
# --- Config ---
BEAUTY_CATEGORY = "Beauty_and_Personal_Care"
DATA_DIR: str = "../data"

# Data split cutoff dates
TRAIN_END_CUTOFF_DATE: str = "2022-08-01"
VALID_END_CUTOFF_DATE: str = "2022-10-01"

# User review count thresholds
USER_MIN_REVIEWS: int = 5
WARM_USER_MIN_REVIEWS: int = 10

# Downsampling for faster iteration
RANDOM_SEED: int = 42
MAX_TRAIN_SIZE: int | None = None
MAX_VALID_SIZE: int | None = None
MAX_TEST_SIZE: int | None = None


## Load reviews


In [3]:
df_reviews = pd.read_csv(f"{DATA_DIR}/reviews.csv")
display(df_reviews.sample(10))
df_reviews.info()


,user_id,parent_asin,rating,timestamp,category
16676332,AEODJZRTCDXQTU4C2KLAWEASCU5A,B08C65BLJX,1.0,1648178254494,Beauty_and_Personal_Care
19333106,AEMTZAAHUCZDJAJI4PWZIAFLBB5Q,B09T6TNZLV,5.0,1655124243602,Clothing_Shoes_and_Jewelry
13557005,AHDPI7JPFPROYR5VIEGVIQNDW53A,B0C6FTDXCV,1.0,1640191828488,Clothing_Shoes_and_Jewelry
20364622,AE4Z4QDGRZ7VCOZE3BWVNHKFQOGQ,B07VQLM41N,1.0,1657727216516,Beauty_and_Personal_Care
11390060,AH5MHZSXGFHQ2RPMZGILHWM4R7BA,B001KYQ7LG,5.0,1634104083197,Beauty_and_Personal_Care
26864586,AHJVOKUIRK5YOQ2LFYIPPQ6DYVDA,B07S2Y3THP,5.0,1672430821980,Clothing_Shoes_and_Jewelry
5305278,AHZFHVPEGBMOM3OFL3N4YACMTQBQ,B07Q2CWRZ7,5.0,1620066896853,Clothing_Shoes_and_Jewelry
24457859,AHQZW46Y7XTUD2DJWKEPT6YLGPYA,B0BSTDFVPT,4.0,1667089166379,Beauty_and_Personal_Care
22424221,AFUKQ7FQ4LLDVAJZLC6NKCPOI4LA,B076VQQ962,2.0,1662238898625,Clothing_Shoes_and_Jewelry
1338421,AFOUS45KXV2376CDYZDJTFRKYCRA,B09L5NL94G,5.0,1612289333771,Beauty_and_Personal_Care


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26876611 entries, 0 to 26876610
Data columns (total 5 columns):
 #   Column       Dtype  
---  ------       -----  
 0   user_id      object 
 1   parent_asin  object 
 2   rating       float64
 3   timestamp    int64  
 4   category     object 
dtypes: float64(1), int64(1), object(3)
memory usage: 1.0+ GB


## Load items


In [4]:
df_items = pd.read_csv(f"{DATA_DIR}/items.csv")
display(df_items.sample(10))
df_items.info()


,parent_asin,title,price,store,category
2369256,B08CXVTHPZ,Gym Sports Bag for Men Women Workout Duffel Ba...,NaN,Talanes,Clothing_Shoes_and_Jewelry
1604586,B099Z53PW8,Womens Round Neck Casual Loose Long Sleeve Tun...,NaN,Cyanstyle,Clothing_Shoes_and_Jewelry
616218,B09CGJDVNV,LucMatton Men's 2 Piece Outfit Long Sleeve But...,52.99,LucMatton,Clothing_Shoes_and_Jewelry
1686173,B08TBDTJGC,Kitinjoy Pillow Soft Slide Cloud Slippers Sand...,NaN,Kitinjoy,Clothing_Shoes_and_Jewelry
451583,B09BZZZRQY,FOYOCER Big Hair Claw Clips Flower Hair Clips ...,NaN,FOYOCER,Beauty_and_Personal_Care
2512802,B0BBZZZLDM,Moon Star Hoop Earrings for Girls - Sterling S...,NaN,EARINGINGS,Clothing_Shoes_and_Jewelry
956404,B08885Y5P7,16G Cartilage Earring-ALL,NaN,Generi,Clothing_Shoes_and_Jewelry
1023810,B09Y8B2GH8,AOMEI Women's 2 Piece Sets V Neck Button Down ...,49.99,AOMEI,Clothing_Shoes_and_Jewelry
1473164,B08KDLPVRX,"Face Cover Bandana, Soft Cotton Fabric Mask Ha...",13.99,UOUDIO,Clothing_Shoes_and_Jewelry
344736,B07HJCQLLJ,CURL KEEPER - Original Multipack - 3 Pack Of C...,NaN,Curl Keeper,Beauty_and_Personal_Care


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2694121 entries, 0 to 2694120
Data columns (total 5 columns):
 #   Column       Dtype  
---  ------       -----  
 0   parent_asin  object 
 1   title        object 
 2   price        float64
 3   store        object 
 4   category     object 
dtypes: float64(1), object(4)
memory usage: 102.8+ MB


## Map user/item IDs to integers


In [5]:
# As RecBole expects integer IDs, we need to create mappings from the original string IDs to integers.
user_ids: set[str] = set(df_reviews["user_id"])
item_ids: set[str] = set(df_reviews["parent_asin"])

user_map: dict[str, int] = {uid: i+1 for i, uid in enumerate(sorted(user_ids))}
item_map: dict[str, int] = {pid: i+1 for i, pid in enumerate(sorted(item_ids))}

In [6]:
df_reviews['uid'] = df_reviews['user_id'].map(user_map)
df_reviews['iid'] = df_reviews['parent_asin'].map(item_map)
df_items['iid'] = df_items['parent_asin'].map(item_map)


## Filter to beauty category


In [7]:
df_reviews = df_reviews[df_reviews["category"] == BEAUTY_CATEGORY]
print(f"Beauty reviews: {len(df_reviews):,}")

Beauty reviews: 7,300,381


## Filter users with too few reviews


In [8]:
before_users = df_reviews["uid"].nunique()
user_counts = df_reviews.groupby("uid").size()
valid_users = user_counts[user_counts >= USER_MIN_REVIEWS].index
df_reviews = df_reviews[df_reviews["uid"].isin(valid_users)]
after_users = df_reviews["uid"].nunique()
print(f"Removed {before_users - after_users:,} users with < {USER_MIN_REVIEWS} reviews "
      f"({after_users:,} users remain, {len(df_reviews):,} reviews)")

Removed 4,221,111 users with < 5 reviews (168,377 users remain, 1,585,905 reviews)


## Split and filter train/valid/test 


In [9]:
def _date_to_ms(date_str: str) -> int:
    return int(pd.Timestamp(date_str, tz="UTC").timestamp() * 1000)

timestamps = sorted(df_reviews["timestamp"].values)
train_start_ts = timestamps[0]
train_end_ts = _date_to_ms(TRAIN_END_CUTOFF_DATE)
valid_end_ts = _date_to_ms(VALID_END_CUTOFF_DATE)
print(f"Train start timestamp: {train_start_ts} ({pd.Timestamp(train_start_ts, unit='ms', tz='UTC')})")
print(f"Train end timestamp: {train_end_ts} ({pd.Timestamp(train_end_ts, unit='ms', tz='UTC')})")
print(f"Valid end timestamp: {valid_end_ts} ({pd.Timestamp(valid_end_ts, unit='ms', tz='UTC')})")


Train start timestamp: 1609459264769 (2021-01-01 00:01:04.769000+00:00)
Train end timestamp: 1659312000000 (2022-08-01 00:00:00+00:00)
Valid end timestamp: 1664582400000 (2022-10-01 00:00:00+00:00)


In [10]:
df_train = df_reviews[df_reviews["timestamp"] <= train_end_ts]
df_valid = df_reviews[(df_reviews["timestamp"] > train_end_ts) & (df_reviews["timestamp"] <= valid_end_ts)]
df_test = df_reviews[df_reviews["timestamp"] > valid_end_ts]

print(f"Train reviews: {len(df_train):,}")
print('Train interactions per user', len(df_train) / len(df_train['uid'].unique()))

if MAX_TRAIN_SIZE is not None and len(df_train) > MAX_TRAIN_SIZE:
    df_train = df_train.sample(n=MAX_TRAIN_SIZE, random_state=RANDOM_SEED)
    print(f"Downsampled train reviews to {len(df_train):,}")
if MAX_VALID_SIZE is not None and len(df_valid) > MAX_VALID_SIZE:
    df_valid = df_valid.sample(n=MAX_VALID_SIZE, random_state=RANDOM_SEED)
    print(f"Downsampled valid reviews to {len(df_valid):,}")
if MAX_TEST_SIZE is not None and len(df_test) > MAX_TEST_SIZE:
    df_test = df_test.sample(n=MAX_TEST_SIZE, random_state=RANDOM_SEED)
    print(f"Downsampled test reviews to {len(df_test):,}")

train_users = set(df_train["uid"].unique())
df_valid = df_valid[df_valid["uid"].isin(train_users)]
df_test = df_test[df_test["uid"].isin(train_users)]
df_all = pd.concat([df_train, df_valid, df_test])

print(f"Valid reviews: {len(df_valid):,}")
print(f"Test reviews: {len(df_test):,}")

display(pd.DataFrame({
    "split": ["train", "valid", "test"],
    "reviews": [len(df_train), len(df_valid), len(df_test)],
    "users": [df_train["uid"].nunique(), df_valid["uid"].nunique(), df_test["uid"].nunique()],
    "items": [df_train["iid"].nunique(), df_valid["iid"].nunique(), df_test["iid"].nunique()],
}))

Train reviews: 1,125,023
Train interactions per user 7.18924255688971
Valid reviews: 139,338
Test reviews: 197,706


,split,reviews,users,items
0,train,1125023,156487,215358
1,valid,139338,47209,52826
2,test,197706,55594,66743


## Define cold vs. warm users with train data


In [11]:
df_train_users = df_train.groupby("uid").size().reset_index(name="num_train")
cold_user_ids: set[int] = set(df_train_users[df_train_users["num_train"] < WARM_USER_MIN_REVIEWS]["uid"])
warm_user_ids: set[int] = set(df_train_users[df_train_users["num_train"] >= WARM_USER_MIN_REVIEWS]["uid"])
print(f"Warm users (>= {WARM_USER_MIN_REVIEWS} reviews): {len(warm_user_ids):,}")
print(f"Cold users (< {WARM_USER_MIN_REVIEWS} reviews): {len(cold_user_ids):,}")


Warm users (>= 10 reviews): 20,100
Cold users (< 10 reviews): 136,387


## Write atomic files


In [12]:
def get_user_category(uid: int) -> int:
    if uid in warm_user_ids:
        return 0  # Warm user
    return 1      # Cold user

def write_user_file(path: Path, uids: set[int]) -> None:
    rows = [(uid, get_user_category(uid)) for uid in uids]
    df_out = pd.DataFrame(rows, columns=["user_id:token", "category:token"])
    df_out.to_csv(path, sep="\t", index=False)
    print(f"Wrote {path} ({len(df_out):,} rows)")

def write_item_file(path: Path, df_items_subset: pd.DataFrame) -> None:
    df_out = df_items_subset[["iid", "title", "store", "price"]].copy()
    df_out["title"] = df_out["title"].fillna("").astype(str).str.replace('"', "", regex=False)
    df_out["store"] = df_out["store"].fillna("").astype(str).str.replace('"', "", regex=False)
    df_out["price"] = pd.to_numeric(df_out["price"], errors="coerce").fillna("")
    df_out.columns = ["item_id:token", "title:token", "store:token", "price:float"]
    df_out.to_csv(path, sep="\t", index=False)
    print(f"Wrote {path} ({len(df_out):,} rows)")

def write_inter_file(path: Path, df: pd.DataFrame) -> None:
    df_out = df[["uid", "iid", "rating", "timestamp"]].copy()
    df_out.columns = ["user_id:token", "item_id:token", "rating:float", "timestamp:float"]
    df_out.to_csv(path, sep="\t", index=False)
    print(f"Wrote {path} ({len(df_out):,} rows)")


In [13]:
dataset_prefix = Path(DATA_DIR) / "beauty" / "beauty"
dataset_prefix.parent.mkdir(parents=True, exist_ok=True)
write_inter_file(dataset_prefix.with_suffix(".train.inter"), df_train)
write_inter_file(dataset_prefix.with_suffix(".valid.inter"), df_valid)
write_inter_file(dataset_prefix.with_suffix(".test.inter"), df_test)
write_user_file(dataset_prefix.with_suffix(".user"), set(df_all['uid'].unique()))
write_item_file(dataset_prefix.with_suffix(".item"), df_items[df_items['iid'].isin(df_all['iid'].unique())])


Wrote ../data/beauty/beauty.train.inter (1,125,023 rows)
Wrote ../data/beauty/beauty.valid.inter (139,338 rows)
Wrote ../data/beauty/beauty.test.inter (197,706 rows)
Wrote ../data/beauty/beauty.user (156,487 rows)
Wrote ../data/beauty/beauty.item (250,852 rows)
